In [ ]:
#Phase 1 — Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
import io

# Upload the Sample Superstore Excel file
uploaded = files.upload()

filename = next(iter(uploaded))

# Load dataset
df = pd.read_excel(io.BytesIO(uploaded[filename]))

# Basic inspection
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
display(df.head())

print("\nData Types and Non-Null Counts:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nStatistical Summary:")
display(df.describe(include="all").T)


In [ ]:
# Check dataset dates and geographic coverage
print("Dataset Date Range:")
print("First Order:", df["Order Date"].min())
print("Last Order:", df["Order Date"].max())

print("\nCountry-wise Records:")
display(df["Country/Region"].value_counts())

print("\nYear-wise Records:")
display(df["Order Date"].dt.year.value_counts().sort_index())

print("\nUnique Orders:", df["Order ID"].nunique())
print("Unique Customers:", df["Customer ID"].nunique())
print("Unique Products:", df["Product ID"].nunique())

# Check possible data-quality issues
print("\nFuture-dated Orders:")
display(df[df["Order Date"] > pd.Timestamp("2026-11-06")][
    ["Order ID", "Order Date", "Ship Date"]
].head(10))

print("\nShip Date Before Order Date:")
display(df[df["Ship Date"] < df["Order Date"]])

print("\nNegative Sales:")
display(df[df["Sales"] < 0])

print("\nDiscount Range:")
print(df["Discount"].min(), "to", df["Discount"].max())

In [ ]:
# 1. Count future-dated records
cutoff_date = pd.Timestamp("2026-11-06")

future_rows = df[df["Order Date"] > cutoff_date]

print("Future-dated rows:", len(future_rows))
print("Future-dated percentage:",
      round(len(future_rows) / len(df) * 100, 2), "%")

# 2. Country-wise date range
country_dates = df.groupby("Country/Region").agg(
    Total_Rows=("Order ID", "count"),
    First_Order=("Order Date", "min"),
    Last_Order=("Order Date", "max"),
    Unique_Customers=("Customer ID", "nunique")
)

display(country_dates)

# 3. Check unique Row IDs
print("Duplicate Row IDs:", df["Row ID"].duplicated().sum())

# 4. Check invalid numerical values
print("Zero or negative sales:", (df["Sales"] <= 0).sum())
print("Zero or negative quantity:", (df["Quantity"] <= 0).sum())
print("Negative discounts:", (df["Discount"] < 0).sum())
print("Discount above 100%:", (df["Discount"] > 1).sum())

# 5. Check country-wise future records
display(
    future_rows["Country/Region"].value_counts()
)

In [ ]:
# Check uploaded file details
print("Uploaded filename:", filename)
print("Total file size (MB):",
      round(len(uploaded[filename]) / (1024 * 1024), 2))

# Check order IDs by year
display(
    df.groupby(df["Order Date"].dt.year)["Order ID"]
      .nunique()
)

In [ ]:
print(df.groupby("Country/Region").size())
print(df.groupby(df["Order Date"].dt.year).size())
print(df.groupby("Country/Region")["Order Date"].agg(["min", "max"]))

In [ ]:
# Keep an untouched copy of the original dataset
df_raw = df.copy()

# Create a separate working copy for cleaning
df_clean = df_raw.copy()

print("Original Shape:", df_raw.shape)
print("Working Copy Shape:", df_clean.shape)

# Check leading/trailing spaces in text columns
text_columns = df_clean.select_dtypes(include="object").columns

print("\nWhitespace Issues:")
for col in text_columns:
    count = df_clean[col].ne(df_clean[col].str.strip()).sum()
    if count > 0:
        print(f"{col}: {count} rows")

# Check blank strings
print("\nBlank String Counts:")
for col in text_columns:
    count = df_clean[col].str.strip().eq("").sum()
    if count > 0:
        print(f"{col}: {count} rows")

# Check date validity
print("\nDate Validation:")
print("Ship date before order date:",
      (df_clean["Ship Date"] < df_clean["Order Date"]).sum())

# Check numerical ranges
print("\nNumerical Validation:")
print("Sales <= 0:", (df_clean["Sales"] <= 0).sum())
print("Quantity <= 0:", (df_clean["Quantity"] <= 0).sum())
print("Discount outside 0–1:",
      ((df_clean["Discount"] < 0) |
       (df_clean["Discount"] > 1)).sum())

# Check category values
for col in ["Segment", "Region", "Category", "Ship Mode"]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())


In [ ]:
#Phase 2: Data Cleaning & Validation
# Inspect actual Postal Code values and their data types
print("Column Data Type:", df_clean["Postal Code"].dtype)

print("\nFirst 15 Postal Codes:")
for value in df_clean["Postal Code"].head(15):
    print(repr(value), type(value))

print("\nUnique Python Data Types:")
print(df_clean["Postal Code"].apply(
    lambda x: type(x).__name__
).value_counts())

print("\nMissing Postal Codes:")
print(df_clean["Postal Code"].isna().sum())

In [ ]:
# Extract postal codes stored as strings
string_postal_codes = df_clean[
    df_clean["Postal Code"].apply(lambda x: isinstance(x, str))
]["Postal Code"]

print("Total String Values:", len(string_postal_codes))

print("\nString Values:")
print(string_postal_codes.value_counts().head(30))

print("\nValues with Leading or Trailing Spaces:")
print([
    repr(x) for x in string_postal_codes
    if x != x.strip()
][:20])

print("\nUnique String Values:")
print(string_postal_codes.nunique())

In [ ]:
# Convert Postal Code to string without losing leading zeros
df_clean["Postal Code"] = (
    df_clean["Postal Code"]
    .astype("string")
    .str.strip()
)

# Verify the conversion
print("New Data Type:", df_clean["Postal Code"].dtype)

print("\nSample Postal Codes:")
display(df_clean[["Country/Region", "Postal Code"]].head(15))

print("\nMissing Values:", df_clean["Postal Code"].isna().sum())

print("\nPostal Code Data Types:")
print(df_clean["Postal Code"].apply(
    lambda x: type(x).__name__
).value_counts())

In [ ]:
# Identify all text columns
text_cols = df_clean.select_dtypes(
    include=["object", "string"]
).columns

# Count values that need trimming
print("Whitespace issues before cleaning:")

for col in text_cols:
    count = (
        df_clean[col].notna() &
        df_clean[col].ne(df_clean[col].str.strip())
    ).sum()

    if count > 0:
        print(col, ":", count)

# Remove unnecessary leading/trailing spaces
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# Verify
print("\nRemaining whitespace issues:")

for col in text_cols:
    count = (
        df_clean[col].notna() &
        df_clean[col].ne(df_clean[col].str.strip())
    ).sum()

    if count > 0:
        print(col, ":", count)

print("\nFinal Shape:", df_clean.shape)
print("Missing Values:", df_clean.isnull().sum().sum())

In [ ]:
# Check final dataset structure
print("Dataset Shape:", df_clean.shape)

# Check missing values and duplicates
print("\nTotal Missing Values:", df_clean.isnull().sum().sum())
print("Duplicate Rows:", df_clean.duplicated().sum())
print("Duplicate Row IDs:", df_clean["Row ID"].duplicated().sum())

# Check data types
print("\nColumn Data Types:")
display(df_clean.dtypes)

# Validate numerical columns
print("\nNumerical Summary:")
display(df_clean[["Sales", "Quantity", "Discount", "Profit"]].describe())

# Check date consistency
print("\nDate Validation:")
print("Invalid Shipping Dates:",
      (df_clean["Ship Date"] < df_clean["Order Date"]).sum())

# Check category consistency
for col in ["Segment", "Region", "Category", "Ship Mode", "Country/Region"]:
    print(f"\n{col}: {df_clean[col].nunique()} unique values")
    print(df_clean[col].unique())

In [ ]:
#Phase 3: Exploratory Data Analysis (EDA)

In [ ]:
# Step 1: Overall Business Performance

import matplotlib.pyplot as plt
import seaborn as sns

# Calculate key business metrics
total_sales = df_clean["Sales"].sum()
total_profit = df_clean["Profit"].sum()
total_orders = df_clean["Order ID"].nunique()
total_customers = df_clean["Customer ID"].nunique()
total_quantity = df_clean["Quantity"].sum()
profit_margin = (total_profit / total_sales) * 100

# Display KPI summary
print("========== BUSINESS PERFORMANCE ==========")
print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Total Quantity Sold: {total_quantity:,}")
print(f"Profit Margin: {profit_margin:.2f}%")

# Create Sales and Profit visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean["Sales"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Sales Distribution")
axes[0].set_xlabel("Sales")

sns.histplot(df_clean["Profit"], bins=50, kde=True, ax=axes[1])
axes[1].set_title("Profit Distribution")
axes[1].set_xlabel("Profit")

plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Category-wise Business Analysis

category_analysis = df_clean.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Total_Orders=("Order ID", "nunique")
).reset_index()

# Calculate profit margin
category_analysis["Profit_Margin_%"] = (
    category_analysis["Total_Profit"] /
    category_analysis["Total_Sales"]
) * 100

# Sort by total sales
category_analysis = category_analysis.sort_values(
    by="Total_Sales", ascending=False
)

display(category_analysis.round(2))

# Visualize category performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=category_analysis,
    x="Category",
    y="Total_Sales",
    ax=axes[0]
)
axes[0].set_title("Category-wise Total Sales")
axes[0].ticklabel_format(style="plain", axis="y")

sns.barplot(
    data=category_analysis,
    x="Category",
    y="Total_Profit",
    ax=axes[1]
)
axes[1].set_title("Category-wise Total Profit")
axes[1].ticklabel_format(style="plain", axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Sub-Category Profitability Analysis

subcategory_analysis = df_clean.groupby(
    ["Category", "Sub-Category"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

# Calculate profit margin
subcategory_analysis["Profit_Margin_%"] = (
    subcategory_analysis["Total_Profit"] /
    subcategory_analysis["Total_Sales"]
) * 100

# Sort by profit
subcategory_analysis = subcategory_analysis.sort_values(
    by="Total_Profit", ascending=True
)

display(subcategory_analysis.round(2))

# Visualize sub-category profitability
plt.figure(figsize=(12, 7))

sns.barplot(
    data=subcategory_analysis,
    x="Total_Profit",
    y="Sub-Category",
    hue="Category"
)

plt.title("Sub-Category-wise Profitability")
plt.xlabel("Total Profit")
plt.ylabel("Sub-Category")
plt.axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Discount Impact on Profitability

# Create discount groups
df_clean["Discount_Band"] = np.select(
    [
        df_clean["Discount"] == 0,
        df_clean["Discount"] <= 0.2,
        df_clean["Discount"] <= 0.4,
        df_clean["Discount"] > 0.4
    ],
    [
        "No Discount",
        "Low Discount",
        "Medium Discount",
        "High Discount"
    ],
    default="Other"
)

# Analyze profitability by discount band
discount_analysis = df_clean.groupby(
    "Discount_Band", observed=True
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Average_Discount=("Discount", "mean")
).reset_index()

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"] /
    discount_analysis["Total_Sales"]
) * 100

display(discount_analysis.round(2))

# Visualize discount vs profit
plt.figure(figsize=(10, 5))

sns.barplot(
    data=discount_analysis,
    x="Discount_Band",
    y="Total_Profit"
)

plt.title("Profit by Discount Band")
plt.xlabel("Discount Level")
plt.ylabel("Total Profit")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Category-wise Discount Impact

category_discount = df_clean.groupby(
    ["Category", "Discount_Band"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

# Calculate profit margin
category_discount["Profit_Margin_%"] = (
    category_discount["Total_Profit"] /
    category_discount["Total_Sales"]
) * 100

display(
    category_discount.sort_values(
        by=["Category", "Total_Profit"]
    ).round(2)
)

# Visualize category-wise profit by discount band
plt.figure(figsize=(12, 6))

sns.barplot(
    data=category_discount,
    x="Discount_Band",
    y="Total_Profit",
    hue="Category"
)

plt.title("Category-wise Profit Across Discount Bands")
plt.xlabel("Discount Band")
plt.ylabel("Total Profit")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Region-wise Business Performance

region_analysis = df_clean.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Total_Quantity=("Quantity", "sum"),
    Total_Customers=("Customer ID", "nunique")
).reset_index()

# Calculate profit margin
region_analysis["Profit_Margin_%"] = (
    region_analysis["Total_Profit"] /
    region_analysis["Total_Sales"]
) * 100

# Sort by profit
region_analysis = region_analysis.sort_values(
    by="Total_Profit", ascending=False
)

display(region_analysis.round(2))

# Visualize regional performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=region_analysis,
    x="Region",
    y="Total_Sales",
    ax=axes[0]
)
axes[0].set_title("Region-wise Sales")

sns.barplot(
    data=region_analysis,
    x="Region",
    y="Total_Profit",
    ax=axes[1]
)
axes[1].set_title("Region-wise Profit")

plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Monthly Sales and Profit Trend

# Set analysis cutoff date
cutoff_date = pd.Timestamp("2026-11-06")

# Separate historical and future-dated records
historical_df = df_clean[
    df_clean["Order Date"] <= cutoff_date
].copy()

future_df = df_clean[
    df_clean["Order Date"] > cutoff_date
].copy()

print("Historical Records:", len(historical_df))
print("Future-Dated Records:", len(future_df))

# Create monthly sales and profit summary
historical_df["Year_Month"] = (
    historical_df["Order Date"].dt.to_period("M").astype(str)
)

monthly_analysis = historical_df.groupby("Year_Month").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

display(monthly_analysis.tail(15).round(2))

# Monthly sales trend
plt.figure(figsize=(15, 5))

sns.lineplot(
    data=monthly_analysis,
    x="Year_Month",
    y="Total_Sales",
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Monthly profit trend
plt.figure(figsize=(15, 5))

sns.lineplot(
    data=monthly_analysis,
    x="Year_Month",
    y="Total_Profit",
    marker="o"
)

plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Total Profit")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Customer-Level Business Analysis

customer_analysis = df_clean.groupby(
    ["Customer ID", "Customer Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

# Calculate customer profit margin
customer_analysis["Profit_Margin_%"] = (
    customer_analysis["Total_Profit"] /
    customer_analysis["Total_Sales"]
) * 100

# Sort customers by sales
top_sales_customers = customer_analysis.sort_values(
    by="Total_Sales", ascending=False
).head(10)

# Sort customers by profit
top_profit_customers = customer_analysis.sort_values(
    by="Total_Profit", ascending=False
).head(10)

print("TOP 10 CUSTOMERS BY SALES")
display(top_sales_customers.round(2))

print("TOP 10 CUSTOMERS BY PROFIT")
display(top_profit_customers.round(2))

In [ ]:
# Step 8.2: Repeat vs One-Time Customer Analysis

customer_frequency = df_clean.groupby("Customer ID").agg(
    Total_Orders=("Order ID", "nunique"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Classify customers by order frequency
customer_frequency["Customer_Type"] = np.where(
    customer_frequency["Total_Orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

# Summarize customer types
frequency_summary = customer_frequency.groupby(
    "Customer_Type"
).agg(
    Total_Customers=("Customer ID", "count"),
    Total_Orders=("Total_Orders", "sum"),
    Total_Sales=("Total_Sales", "sum"),
    Total_Profit=("Total_Profit", "sum")
).reset_index()

# Calculate revenue contribution
frequency_summary["Sales_Contribution_%"] = (
    frequency_summary["Total_Sales"] /
    frequency_summary["Total_Sales"].sum()
) * 100

display(frequency_summary.round(2))

# Visualize customer distribution
plt.figure(figsize=(8, 5))

sns.countplot(
    data=customer_frequency,
    x="Customer_Type"
)

plt.title("Repeat vs One-Time Customers")
plt.xlabel("Customer Type")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

In [ ]:
# Step 8.3: Verify Order-Customer Relationship

orders_per_customer = df_clean.groupby(
    "Order ID"
)["Customer ID"].nunique()

multi_customer_orders = orders_per_customer[
    orders_per_customer > 1
]

print("Total Unique Orders:", df_clean["Order ID"].nunique())

print(
    "Orders Linked to Multiple Customers:",
    len(multi_customer_orders)
)

print("\nOrder IDs with Multiple Customers:")
display(multi_customer_orders)

In [ ]:
# Step 8.4: Inspect Orders Linked to Multiple Customers

problem_orders = ["CA-2025-121465", "CA-2026-130494"]

display(
    df_clean[
        df_clean["Order ID"].isin(problem_orders)
    ][[
        "Order ID",
        "Customer ID",
        "Customer Name",
        "Order Date",
        "Product ID",
        "Sales",
        "Profit"
    ]].sort_values(["Order ID", "Customer ID"])
)

In [ ]:
# Step 8.5: Check Customer Name and ID Mapping

name_id_check = df_clean.groupby(
    "Customer Name"
)["Customer ID"].nunique()

multiple_ids = name_id_check[name_id_check > 1]

print("Customer Names with Multiple IDs:", len(multiple_ids))

display(multiple_ids.sort_values(ascending=False))

In [ ]:
# Step 8.6: Inspect All Records for Harry Olson

display(
    df_clean[
        df_clean["Customer Name"] == "Harry Olson"
    ][[
        "Customer ID",
        "Order ID",
        "Order Date",
        "Sales",
        "Profit"
    ]].sort_values("Order Date")
)

In [ ]:
# Step 9.1: Prepare Historical Data for RFM

rfm_df = df_clean[
    df_clean["Order Date"] <= cutoff_date
].copy()

reference_date = cutoff_date + pd.Timedelta(days=1)

print("RFM Dataset Shape:", rfm_df.shape)
print("Reference Date:", reference_date)
print("Unique Customers:", rfm_df["Customer ID"].nunique())
print("Unique Orders:", rfm_df["Order ID"].nunique())
print("Future Records Excluded:", len(future_df))

In [ ]:
# Step 9.2: Calculate RFM Metrics

rfm = rfm_df.groupby("Customer ID").agg(
    Last_Order_Date=("Order Date", "max"),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Calculate Recency in days
rfm["Recency"] = (
    reference_date - rfm["Last_Order_Date"]
).dt.days

# Arrange RFM columns
rfm = rfm[
    [
        "Customer ID",
        "Last_Order_Date",
        "Recency",
        "Frequency",
        "Monetary",
        "Total_Profit"
    ]
]

print("RFM Dataset Shape:", rfm.shape)

display(rfm.head(10))

print("\nRFM Statistical Summary:")
display(rfm[["Recency", "Frequency", "Monetary", "Total_Profit"]].describe().round(2))

In [ ]:
# Step 9.3: Assign RFM Scores

# Recency: Lower days = Better score
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: Higher orders = Better score
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: Higher sales = Better score
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM Score
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

display(rfm.head(10))

print("\nRFM Score Distribution:")
display(
    rfm[["R_Score", "F_Score", "M_Score"]]
    .apply(pd.Series.value_counts)
    .fillna(0)
    .astype(int)
)

In [ ]:
# Step 9.4: Create Customer Segments

def segment_customer(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"

    elif row["R_Score"] >= 3 and row["F_Score"] >= 4:
        return "Loyal Customers"

    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New Customers"

    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"

    elif row["R_Score"] <= 2 and row["F_Score"] <= 2:
        return "Lost Customers"

    else:
        return "Potential Loyalists"


rfm["Customer_Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

# Segment Summary
segment_summary = rfm.groupby("Customer_Segment").agg(
    Total_Customers=("Customer ID", "count"),
    Total_Sales=("Monetary", "sum"),
    Total_Profit=("Total_Profit", "sum"),
    Average_Recency=("Recency", "mean"),
    Average_Frequency=("Frequency", "mean")
).reset_index()

# Calculate customer and sales contribution
segment_summary["Customer_Contribution_%"] = (
    segment_summary["Total_Customers"] /
    segment_summary["Total_Customers"].sum()
) * 100

segment_summary["Sales_Contribution_%"] = (
    segment_summary["Total_Sales"] /
    segment_summary["Total_Sales"].sum()
) * 100

display(segment_summary.round(2))

In [ ]:
# Step 9.5: Customer Segment Distribution

plt.figure(figsize=(10, 6))

sns.barplot(
    data=segment_summary.sort_values(
        "Total_Customers", ascending=False
    ),
    x="Customer_Segment",
    y="Total_Customers"
)

plt.title("Customer Distribution by RFM Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Step 9.6: Sales Contribution by RFM Segment

plt.figure(figsize=(10, 6))

sns.barplot(
    data=segment_summary.sort_values(
        "Sales_Contribution_%", ascending=False
    ),
    x="Customer_Segment",
    y="Sales_Contribution_%"
)

plt.title("Sales Contribution by RFM Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Sales Contribution (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Step 9.7: Profit Contribution by RFM Segment

segment_summary["Profit_Margin_%"] = (
    segment_summary["Total_Profit"] /
    segment_summary["Total_Sales"]
) * 100

segment_summary["Profit_Contribution_%"] = (
    segment_summary["Total_Profit"] /
    segment_summary["Total_Profit"].sum()
) * 100

display(
    segment_summary[
        [
            "Customer_Segment",
            "Total_Customers",
            "Total_Sales",
            "Total_Profit",
            "Profit_Margin_%",
            "Profit_Contribution_%"
        ]
    ].sort_values(
        "Total_Profit", ascending=False
    ).round(2)
)

In [ ]:
# Step 9.8: Profit Contribution by Customer Segment

plt.figure(figsize=(10, 6))

sns.barplot(
    data=segment_summary.sort_values(
        "Profit_Contribution_%", ascending=False
    ),
    x="Customer_Segment",
    y="Profit_Contribution_%"
)

plt.title("Profit Contribution by RFM Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Profit Contribution (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Step 9.8: Profit Contribution by Customer Segment

plt.figure(figsize=(10, 6))

sns.barplot(
    data=segment_summary.sort_values(
        "Profit_Contribution_%", ascending=False
    ),
    x="Customer_Segment",
    y="Profit_Contribution_%"
)

plt.title("Profit Contribution by RFM Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Profit Contribution (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Step 9.9: RFM Validation

print("Total RFM Customers:", len(rfm))
print("Unique Customer IDs:", rfm["Customer ID"].nunique())

print("\nMissing Values:")
display(rfm.isnull().sum())

print("\nCustomer Segment Counts:")
display(rfm["Customer_Segment"].value_counts())

print("\nSales Validation:")
print("RFM Total Sales:", round(rfm["Monetary"].sum(), 2))
print("Historical Sales:", round(rfm_df["Sales"].sum(), 2))

print("\nProfit Validation:")
print("RFM Total Profit:", round(rfm["Total_Profit"].sum(), 2))
print("Historical Profit:", round(rfm_df["Profit"].sum(), 2))